# Sesión 2 — Ingesta de datos y capa Bronze  
## Notebook del instructor actualizado con archivos reales

**Curso:** Databricks práctico — Lumi Commerce Lakehouse + Caso espejo Lluvia/Caña/Bagazo  
**Rol:** versión del instructor con soluciones completas, validaciones reales y troubleshooting.

### Objetivo
Cargar los archivos reales de Lumi Commerce y Bagazo en Databricks Free Edition, crear esquemas Bronze, materializar tablas Delta administradas y construir una tabla de auditoría de carga.

### Ajuste importante de esta versión
Este notebook ya usa los nombres reales de archivos, columnas y conteos esperados de los datasets adjuntos:

- 9 archivos CSV de Lumi Commerce.
- 1 archivo Excel de Bagazo convertido a CSV para facilitar la carga con Spark.

> En Databricks Free Edition vamos a trabajar de forma controlada desde archivos cargados al workspace, DBFS/FileStore o repositorio. En Azure Databricks empresarial, la misma lógica normalmente apuntaría a ADLS Gen2, Volumes gobernados por Unity Catalog, Auto Loader y Jobs productivos.

## 0. Inventario real de archivos

### Lumi Commerce

| Tabla lógica | Archivo | Filas esperadas | Columnas |
|---|---|---:|---:|
| `customers` | `olist_customers_dataset.csv` | 99,441 | 5 |
| `orders` | `olist_orders_dataset.csv` | 99,441 | 8 |
| `order_items` | `olist_order_items_dataset.csv` | 112,650 | 7 |
| `order_payments` | `olist_order_payments_dataset.csv` | 103,886 | 5 |
| `order_reviews` | `olist_order_reviews_dataset.csv` | 99,224 | 7 |
| `products` | `olist_products_dataset.csv` | 32,951 | 9 |
| `sellers` | `olist_sellers_dataset.csv` | 3,095 | 4 |
| `geolocation` | `olist_geolocation_dataset.csv` | 1,000,163 | 5 |
| `product_category_translation` | `product_category_name_translation.csv` | 71 | 2 |

### Bagazo

| Dataset | Archivo original | CSV recomendado | Filas | Columnas | Rango de fechas |
|---|---|---|---:|---:|---|
| `molienda_bagazo_y_lluvias` | `molienda_bagazo_y_lluvias_II (1).xlsx` | `molienda_bagazo_y_lluvias_II.csv` | 798 | 13 | 2024-01-01 a 2026-03-08 |

## 1. Preguntas para abrir la práctica

1. ¿Qué diferencia hay entre cargar un archivo y gobernar un dato?
2. ¿Qué problema aparece si cada analista carga su propio CSV sin auditoría?
3. ¿Por qué no conviene limpiar todo directamente en la primera carga?
4. ¿Qué evidencia mínima pediría una gerencia para confiar en la carga?
5. ¿Qué riesgos aparecen si no verificamos columnas y conteos esperados?

In [0]:
# =========================================
# 1. Validación del entorno
# =========================================
from datetime import datetime
from typing import Dict, List, Optional
import re
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

print("Spark version:", spark.version)

current_catalog = spark.sql("SELECT current_catalog() AS catalog").collect()[0]["catalog"]
current_database = spark.sql("SELECT current_database() AS database").collect()[0]["database"]

print("Current catalog:", current_catalog)
print("Current database:", current_database)

## 2. Parámetros de trabajo

Ajusta las rutas según dónde cargaste los archivos.

### Opción recomendada para clase

**Repositorio conectado con Databricks**
```python
LUMI_RAW_PATH = "file:/Workspace/Repos/<usuario>/carvajaldatabricks/data/raw/lumi"
BAGAZO_RAW_PATH = "file:/Workspace/Repos/<usuario>/carvajaldatabricks/data/raw/bagazo"
```

### Alternativa rápida

**DBFS/FileStore**
```python
LUMI_RAW_PATH = "dbfs:/FileStore/lumi"
BAGAZO_RAW_PATH = "dbfs:/FileStore/bagazo"
```

> Nota: el archivo original de Bagazo venía en Excel. Para esta sesión se entrega una versión CSV llamada `molienda_bagazo_y_lluvias_II.csv`, porque Spark lee CSV de forma nativa y es más estable para clase.

In [0]:
# =========================================
# 2. Parámetros ajustables
# =========================================
CATALOG = current_catalog

LUMI_BRONZE_SCHEMA = "lumi_bronze"
BAGAZO_BRONZE_SCHEMA = "bagazo_bronze"
CONTROL_SCHEMA = "control"

# Ajustar estas rutas de acuerdo con el lugar real donde cargaste los archivos.
LUMI_RAW_PATH = "/Volumes/workspace/raw/lumi"
BAGAZO_RAW_PATH = "/Volumes/workspace/raw/bagazo"

print("LUMI_RAW_PATH:", LUMI_RAW_PATH)
print("BAGAZO_RAW_PATH:", BAGAZO_RAW_PATH)

display(dbutils.fs.ls(LUMI_RAW_PATH))
display(dbutils.fs.ls(BAGAZO_RAW_PATH))

# Archivos reales de Lumi Commerce.
LUMI_FILES = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv"
}

# Archivo recomendado de Bagazo ya convertido a CSV.
BAGAZO_FILE = "molienda_bagazo_y_lluvias_II.csv"
BAGAZO_DATASET = "molienda_bagazo_y_lluvias_II"

EXPECTED_ROWS = globals().get("EXPECTED_ROWS", {})
EXPECTED_COLUMNS = globals().get("EXPECTED_COLUMNS", {})

EXPECTED_ROWS[BAGAZO_DATASET] = 798

EXPECTED_COLUMNS[BAGAZO_DATASET] = [
    "fecha",
    "promedio_lluvias_providencia_mm",
    "cana_molida_providencia_toneladas",
    "bagazo_entregado_providencia_toneladas",
    "comentarios_providencia",
    "promedio_lluvias_ilc_mm",
    "cana_molida_ilc_toneladas",
    "bagazo_entregado_ilc_toneladas",
    "comentarios_ilc",
    "promedio_lluvias_incauca_mm",
    "cana_molida_incauca_toneladas",
    "bagazo_entregado_incauca_toneladas",
    "comentarios_incauca"
]

# Columnas esperadas tomadas de los archivos reales.
EXPECTED_COLUMNS = {
    "customers": [
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ],
    "orders": [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "order_items": [
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "shipping_limit_date",
        "price",
        "freight_value"
    ],
    "order_payments": [
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value"
    ],
    "order_reviews": [
        "review_id",
        "order_id",
        "review_score",
        "review_comment_title",
        "review_comment_message",
        "review_creation_date",
        "review_answer_timestamp"
    ],
    "products": [
        "product_id",
        "product_category_name",
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ],
    "sellers": [
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    ],
    "geolocation": [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state"
    ],
    "product_category_translation": [
        "product_category_name",
        "product_category_name_english"
    ],
    "molienda_bagazo_y_lluvias": [
        "Fecha",
        "PROMEDIO LLUVIAS  PROVIDENCIA (mm)",
        "CAÑA MOLIDA PROVIDENCIA (Toneladas)",
        "BAGAZO ENTREGADO PROVIDENCIA (Toneladas)",
        "Comentarios\nPROVIDENCIA",
        "PROMEDIO LLUVIAS ILC (mm)",
        "CAÑA MOLIDA ILC (Toneladas)",
        "BAGAZO ENTREGADO ILC (Toneladas)",
        "Comentarios\nILC",
        "PROMEDIO LLUVIAS INCAUCA  (mm)",
        "CAÑA MOLIDA INCAUCA (Toneladas)",
        "BAGAZO ENTREGADO INCAUCA (Toneladas)",
        "Comentarios\nINCAUCA"
    ]
}

print("CATALOG:", CATALOG)
print("LUMI_RAW_PATH:", LUMI_RAW_PATH)
print("BAGAZO_RAW_PATH:", BAGAZO_RAW_PATH)

## 3. Crear esquemas Bronze y esquema de control

Creamos tres esquemas:

- `lumi_bronze`: datos crudos del proyecto Lumi.
- `bagazo_bronze`: datos crudos del caso espejo Bagazo.
- `control`: tablas de auditoría y control técnico.

En Free Edition esto nos da una práctica simple y reproducible. En Azure Databricks empresarial, estos esquemas estarían gobernados por Unity Catalog con permisos, ownership y políticas de acceso.

In [0]:
# =========================================
# 3. Crear esquemas
# =========================================
for schema in [LUMI_BRONZE_SCHEMA, BAGAZO_BRONZE_SCHEMA, CONTROL_SCHEMA]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{schema}`")
    print(f"Schema listo: {CATALOG}.{schema}")

## 4. Funciones auxiliares

Estas funciones permiten:

1. Normalizar nombres técnicos de columnas.
2. Leer CSV con opciones estándar.
3. Guardar una tabla Delta administrada.
4. Validar columnas y conteos esperados.
5. Registrar auditoría de carga.

La normalización es conservadora: baja a minúsculas, elimina tildes básicas y reemplaza caracteres no alfanuméricos por `_`.

In [0]:
# =========================================
# 4. Funciones auxiliares
# =========================================
def normalize_column_name(name: str) -> str:
    replacements = {
        "á": "a", "é": "e", "í": "i", "ó": "o", "ú": "u", "ñ": "n",
        "Á": "a", "É": "e", "Í": "i", "Ó": "o", "Ú": "u", "Ñ": "n"
    }
    for old, new in replacements.items():
        name = name.replace(old, new)
    name = name.replace("\n", "_").replace("\r", "_")
    name = name.strip().lower()
    name = re.sub(r"[^a-zA-Z0-9_]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name


def normalize_columns(df):
    new_cols = [normalize_column_name(c) for c in df.columns]
    return df.toDF(*new_cols)


def read_csv_safely(path: str, sep: str = ",", infer_schema: bool = True, normalize_cols: bool = True):
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", infer_schema)
        .option("sep", sep)
        .option("multiLine", True)
        .option("quote", '"')
        .option("escape", '"')
        .csv(path)
    )
    if normalize_cols:
        df = normalize_columns(df)
    return df


def normalized_expected_columns(dataset_name: str):
    return [normalize_column_name(c) for c in EXPECTED_COLUMNS[dataset_name]]


def validate_columns(df, dataset_name: str):
    expected = set(normalized_expected_columns(dataset_name))
    actual = set(df.columns)
    missing = sorted(list(expected - actual))
    extra = sorted(list(actual - expected))
    if missing:
        print(f"⚠️ Columnas faltantes en {dataset_name}: {missing}")
    if extra:
        print(f"ℹ️ Columnas adicionales en {dataset_name}: {extra}")
    if not missing:
        print(f"✅ Columnas esperadas presentes en {dataset_name}")
    return missing, extra


def write_delta_table(df, table_fqn: str, mode: str = "overwrite"):
    (
        df.write
        .format("delta")
        .mode(mode)
        .option("overwriteSchema", "true")
        .saveAsTable(table_fqn)
    )
    print(f"Tabla escrita: {table_fqn}")


audit_rows = []


def register_audit(dataset: str, archivo_origen: str, tabla_destino: str, filas_cargadas: Optional[int], columnas: Optional[int], estado: str, observaciones: str):
    audit_rows.append({
        "dataset": dataset,
        "archivo_origen": archivo_origen,
        "tabla_destino": tabla_destino,
        "filas_cargadas": int(filas_cargadas) if filas_cargadas is not None else None,
        "columnas": int(columnas) if columnas is not None else None,
        "fecha_carga": datetime.now().isoformat(timespec="seconds"),
        "estado": estado,
        "observaciones": observaciones
    })

## 5. Ingesta de Lumi Commerce a Bronze

Cargaremos los 9 archivos reales de Lumi.

**Punto pedagógico:** en Bronze no vamos a resolver calidad de datos de negocio. Pero sí verificamos que:
- El archivo existe.
- Las columnas esperadas están presentes.
- El conteo se puede comparar contra el conteo esperado.
- La tabla destino queda creada.

In [0]:
# =========================================
# 5. Ingesta Lumi
# =========================================
loaded_lumi = {}

for dataset, filename in LUMI_FILES.items():
    path = f"{LUMI_RAW_PATH}/{filename}"
    table_fqn = f"`{CATALOG}`.`{LUMI_BRONZE_SCHEMA}`.`{dataset}`"

    print("=" * 100)
    print(f"Cargando dataset: {dataset}")
    print(f"Archivo: {path}")
    print(f"Destino: {table_fqn}")

    try:
        df = read_csv_safely(path)
        missing, extra = validate_columns(df, dataset)
        rows = df.count()
        cols = len(df.columns)

        expected_rows = EXPECTED_ROWS[dataset]
        if rows == expected_rows:
            row_msg = f"Conteo OK: {rows:,} filas"
            print(f"✅ {row_msg}")
        else:
            row_msg = f"Conteo diferente. Esperado: {expected_rows:,}; obtenido: {rows:,}"
            print(f"⚠️ {row_msg}")

        write_delta_table(df, table_fqn)
        loaded_lumi[dataset] = df

        estado = "OK" if not missing and rows == expected_rows else "REVISAR"
        observaciones = f"{row_msg}. Columnas: {cols}. Faltantes: {missing}. Extras: {extra}"
        register_audit(dataset, filename, f"{CATALOG}.{LUMI_BRONZE_SCHEMA}.{dataset}", rows, cols, estado, observaciones)

    except Exception as e:
        print(f"❌ Error cargando {dataset}: {e}")
        register_audit(dataset, filename, f"{CATALOG}.{LUMI_BRONZE_SCHEMA}.{dataset}", None, None, "ERROR", str(e))

## 6. Validaciones iniciales de Lumi

Estas validaciones no reemplazan la limpieza de Silver. Sirven para verificar que la ingesta fue técnicamente correcta y para formar criterio de datos.

In [0]:
# =========================================
# 6. Resumen de tablas Lumi cargadas
# =========================================
summary_rows = []

for dataset in LUMI_FILES.keys():
    table_fqn = f"`{CATALOG}`.`{LUMI_BRONZE_SCHEMA}`.`{dataset}`"
    df = spark.table(table_fqn)
    rows = df.count()
    cols = len(df.columns)
    expected = EXPECTED_ROWS[dataset]
    summary_rows.append((dataset, rows, expected, cols, "OK" if rows == expected else "REVISAR"))

summary_df = spark.createDataFrame(summary_rows, ["dataset", "filas_obtenidas", "filas_esperadas", "columnas", "estado"])
display(summary_df.orderBy("dataset"))

In [0]:
# =========================================
# 7. Llaves candidatas y relaciones iniciales
# =========================================
candidate_keys = {
    "customers": ["customer_id", "customer_unique_id"],
    "orders": ["order_id", "customer_id"],
    "order_items": ["order_id", "order_item_id", "product_id", "seller_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id", "order_id"],
    "products": ["product_id", "product_category_name"],
    "sellers": ["seller_id"],
    "geolocation": ["geolocation_zip_code_prefix"],
    "product_category_translation": ["product_category_name"],
}

key_summary = []

for dataset, keys in candidate_keys.items():
    df = spark.table(f"`{CATALOG}`.`{LUMI_BRONZE_SCHEMA}`.`{dataset}`")
    total = df.count()
    for key in keys:
        if key in df.columns:
            distinct = df.select(key).distinct().count()
            nulls = df.filter(F.col(key).isNull()).count()
            key_summary.append((dataset, key, total, distinct, nulls, distinct == total and nulls == 0))

key_summary_df = spark.createDataFrame(
    key_summary,
    ["dataset", "columna", "filas", "distintos", "nulos", "parece_pk_unica"]
)
display(key_summary_df.orderBy("dataset", "columna"))

### Discusión con el grupo

1. ¿Por qué `customer_id` parece único pero `customer_unique_id` no necesariamente?
2. ¿Por qué `order_items` no tiene una sola fila por pedido?
3. ¿Por qué `geolocation_zip_code_prefix` no es único?
4. ¿Qué tablas parecen dimensión y cuáles parecen hechos?

## 7. Ingesta del caso Bagazo

El archivo original venía en Excel: `molienda_bagazo_y_lluvias_II (1).xlsx`.

Para Databricks Free Edition y para evitar problemas de dependencias de Excel, se recomienda cargar la versión CSV:

`molienda_bagazo_y_lluvias_II.csv`

**Granularidad inicial:** una fila por fecha, con columnas separadas por ingenio para lluvia, caña molida, bagazo entregado y comentarios. En la sesión 3 o 4 lo transformaremos a formato largo.

In [0]:
import re
import unicodedata
from pyspark.sql import functions as F

def normalize_column_name(col_name: str) -> str:
    col_name = str(col_name)
    col_name = col_name.replace("\n", " ")
    col_name = unicodedata.normalize("NFKD", col_name)
    col_name = col_name.encode("ascii", "ignore").decode("ascii")
    col_name = col_name.lower().strip()
    col_name = re.sub(r"[^a-z0-9]+", "_", col_name)
    col_name = re.sub(r"_+", "_", col_name)
    col_name = col_name.strip("_")
    return col_name

def normalize_columns(df):
    for old_col in df.columns:
        df = df.withColumnRenamed(old_col, normalize_column_name(old_col))
    return df

def drop_empty_columns(df):
    non_empty_exprs = [
        F.count(
            F.when(
                F.col(c).isNotNull() & (F.trim(F.col(c).cast("string")) != ""),
                c
            )
        ).alias(c)
        for c in df.columns
    ]

    counts = df.agg(*non_empty_exprs).collect()[0].asDict()

    cols_to_keep = [
        c for c, count_value in counts.items()
        if count_value > 0
    ]

    cols_dropped = [
        c for c, count_value in counts.items()
        if count_value == 0
    ]

    print("Columnas vacías eliminadas:", cols_dropped)

    return df.select(cols_to_keep)

def read_csv_safely(path):
    candidates = []

    for sep in [",", ";", "\t", "|"]:
        temp_df = (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
            .option("sep", sep)
            .option("multiLine", True)
            .option("quote", '"')
            .option("escape", '"')
            .csv(path)
        )
        candidates.append((sep, len(temp_df.columns), temp_df))

    best_sep, best_cols, best_df = max(candidates, key=lambda x: x[1])

    print(f"Separador detectado: '{best_sep}'")
    print(f"Columnas detectadas antes de limpieza: {best_cols}")

    best_df = normalize_columns(best_df)
    best_df = drop_empty_columns(best_df)

    print(f"Columnas finales: {len(best_df.columns)}")
    print("Columnas leídas:")
    print(best_df.columns)

    return best_df

In [0]:
# =========================================
# 8. Ingesta Bagazo
# =========================================
agazo_dataset = BAGAZO_DATASET
bagazo_path = f"{BAGAZO_RAW_PATH}/{BAGAZO_FILE}"
bagazo_table_fqn = f"`{CATALOG}`.`{BAGAZO_BRONZE_SCHEMA}`.`{bagazo_dataset}`"

print(f"Cargando Bagazo desde: {bagazo_path}")

try:
    bagazo_df = read_csv_safely(bagazo_path)

    missing, extra = validate_columns(bagazo_df, bagazo_dataset)

    rows = bagazo_df.count()
    cols = len(bagazo_df.columns)

    expected_rows = EXPECTED_ROWS.get(bagazo_dataset)

    if expected_rows is not None and rows == expected_rows:
        row_msg = f"Conteo OK: {rows:,} filas"
        print(f"✅ {row_msg}")
    elif expected_rows is not None:
        row_msg = f"Conteo diferente. Esperado: {expected_rows:,}; obtenido: {rows:,}"
        print(f"⚠️ {row_msg}")
    else:
        row_msg = f"Conteo obtenido: {rows:,} filas"
        print(f"ℹ️ {row_msg}")

    write_delta_table(bagazo_df, bagazo_table_fqn)

    estado = "OK" if not missing and (expected_rows is None or rows == expected_rows) else "REVISAR"

    register_audit(
        bagazo_dataset,
        BAGAZO_FILE,
        f"{CATALOG}.{BAGAZO_BRONZE_SCHEMA}.{bagazo_dataset}",
        rows,
        cols,
        estado,
        f"{row_msg}. Faltantes: {missing}. Extras: {extra}"
    )

    print(f"✅ Tabla Bronze creada: {CATALOG}.{BAGAZO_BRONZE_SCHEMA}.{bagazo_dataset}")
    display(bagazo_df.limit(10))

except Exception as e:
    print(f"❌ Error cargando Bagazo: {e}")

    register_audit(
        bagazo_dataset,
        BAGAZO_FILE,
        f"{CATALOG}.{BAGAZO_BRONZE_SCHEMA}.{bagazo_dataset}",
        None,
        None,
        "ERROR",
        str(e)
    )

In [0]:
# =========================================
# 9. Validaciones iniciales Bagazo
# =========================================
bagazo_df = spark.table(bagazo_table_fqn)

display(bagazo_df.limit(10))

# La columna 'fecha' viene normalizada desde 'Fecha'.
if "fecha" in bagazo_df.columns:
    bagazo_dates = (
        bagazo_df
        .select(F.to_date("fecha").alias("fecha"))
        .agg(
            F.min("fecha").alias("fecha_min"),
            F.max("fecha").alias("fecha_max"),
            F.count("*").alias("filas")
        )
    )
    display(bagazo_dates)
else:
    print("⚠️ No se encontró columna fecha después de normalizar.")

In [0]:
# =========================================
# 10. Validación de columnas por ingenio
# =========================================
bagazo_columns = bagazo_df.columns

for ingenio in ["providencia", "ilc", "incauca"]:
    cols_ingenio = [c for c in bagazo_columns if ingenio in c]
    print(f"\nIngenio: {ingenio.upper()}")
    for c in cols_ingenio:
        print(" -", c)

## 8. Crear tabla de auditoría

La auditoría permite responder preguntas mínimas:

- ¿Qué dataset cargué?
- ¿Desde qué archivo?
- ¿Hacia qué tabla?
- ¿Cuántas filas y columnas quedaron?
- ¿Cuándo se cargó?
- ¿La carga quedó OK, en revisión o falló?
- ¿Qué observación dejó el proceso?

In [0]:
# =========================================
# 11. Materializar auditoría
# =========================================
audit_df = spark.createDataFrame(audit_rows)

audit_table_fqn = f"`{CATALOG}`.`{CONTROL_SCHEMA}`.`audit_ingestion_sesion_02`"

(
    audit_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(audit_table_fqn)
)

display(spark.table(audit_table_fqn).orderBy("dataset"))

## 9. Consultas SQL de validación

A partir de aquí puedes alternar entre PySpark y SQL. Esto ayuda a los analistas y perfiles de gobierno de datos a revisar evidencia con un lenguaje conocido.

In [0]:
# =========================================
# 12. SQL desde notebook Python
# =========================================
display(spark.sql(f"""
SELECT dataset, archivo_origen, filas_cargadas, columnas, estado, fecha_carga
FROM `{CATALOG}`.`{CONTROL_SCHEMA}`.`audit_ingestion_sesion_02`
ORDER BY dataset
"""))

In [0]:
# =========================================
# 13. Validación rápida de estados
# =========================================
display(spark.sql(f"""
SELECT estado, COUNT(*) AS cantidad_datasets
FROM `{CATALOG}`.`{CONTROL_SCHEMA}`.`audit_ingestion_sesion_02`
GROUP BY estado
ORDER BY estado
"""))

## 10. Errores comunes y troubleshooting

### Error: archivo no encontrado
**Causa probable:** `LUMI_RAW_PATH` o `BAGAZO_RAW_PATH` apunta a una ruta diferente.  
**Solución:** revisar dónde fueron cargados los archivos y ajustar las variables.

### Error: columnas faltantes
**Causa probable:** archivo equivocado, separador equivocado, encabezado mal leído o cambio de nombre.  
**Solución:** revisar `df.columns`, validar separador y comparar contra el perfil de datos.

### Error: conteo diferente
**Causa probable:** archivo truncado, archivo duplicado, filtro accidental o versión diferente del dataset.  
**Solución:** comparar tamaño del archivo y volver a cargar desde la fuente original.

### Error con Excel de Bagazo
**Causa probable:** Spark no lee `.xlsx` de forma nativa.  
**Solución:** usar el CSV convertido `molienda_bagazo_y_lluvias_II.csv`.

## 11. Cierre de la sesión

### Lo que debe quedar listo

- `lumi_bronze.customers`
- `lumi_bronze.orders`
- `lumi_bronze.order_items`
- `lumi_bronze.order_payments`
- `lumi_bronze.order_reviews`
- `lumi_bronze.products`
- `lumi_bronze.sellers`
- `lumi_bronze.geolocation`
- `lumi_bronze.product_category_translation`
- `bagazo_bronze.molienda_bagazo_y_lluvias`
- `control.audit_ingestion_sesion_02`

### Pregunta final
Si mañana un gerente pregunta “¿puedo confiar en que los datos cargaron bien?”, ¿qué evidencia mostrarías?